# Retail Sales Exploration
**Student:** Juan Carlos Ferrer Jr
**Course:** CAP4784 Big Data
 
## Business question
Which store and product category generate the most revenue?


In [0]:
%sql
SELECT current_catalog(), current_schema();


current_catalog(),current_schema()
workspace,default


In [0]:
%sql
USE CATALOG `workspace`;
CREATE SCHEMA IF NOT EXISTS module1_retail;
USE SCHEMA module1_retail;
CREATE VOLUME IF NOT EXISTS raw_files;

In [0]:
file_path = (
    "/Volumes/workspace/module1_retail/"
    "raw_files/transactions.csv"
)
 
schema = """
transaction_id INT,
sale_date DATE,
store STRING,
category STRING,
quantity INT,
unit_price DECIMAL(10,2)
"""
 
sales_df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(file_path)
)
display(sales_df)


transaction_id,sale_date,store,category,quantity,unit_price
1,2026-10-01,Miami,Electronics,2,100.00
2,2026-10-01,Kendall,Home,3,40.00
3,2026-10-01,Hialeah,Sports,4,25.00
4,2026-10-02,Miami,Home,2,50.00
5,2026-10-02,Kendall,Electronics,1,200.00
6,2026-10-02,Hialeah,Electronics,2,150.00
7,2026-10-03,Miami,Sports,5,20.00
8,2026-10-03,Kendall,Sports,2,50.00
9,2026-10-03,Hialeah,Home,4,30.00
10,2026-10-04,Miami,Electronics,1,300.00


In [0]:
sales_df.printSchema()
print("Number of transactions:", sales_df.count())


root
 |-- transaction_id: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- store: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)

Number of transactions: 12


In [0]:
from pyspark.sql import functions as F
 
sales_with_revenue = sales_df.withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
)
display(sales_with_revenue)


transaction_id,sale_date,store,category,quantity,unit_price,revenue
1,2026-10-01,Miami,Electronics,2,100.00,200.00
2,2026-10-01,Kendall,Home,3,40.00,120.00
3,2026-10-01,Hialeah,Sports,4,25.00,100.00
4,2026-10-02,Miami,Home,2,50.00,100.00
5,2026-10-02,Kendall,Electronics,1,200.00,200.00
6,2026-10-02,Hialeah,Electronics,2,150.00,300.00
7,2026-10-03,Miami,Sports,5,20.00,100.00
8,2026-10-03,Kendall,Sports,2,50.00,100.00
9,2026-10-03,Hialeah,Home,4,30.00,120.00
10,2026-10-04,Miami,Electronics,1,300.00,300.00


In [0]:
(
    sales_with_revenue.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("module1_retail.sales")
)


In [0]:
%sql
-- Question A
SELECT 
    COUNT(transaction_id) AS total_transactions,
    SUM(quantity) AS total_units_sold,
    SUM(revenue) AS total_revenue
FROM module1_retail.sales;

total_transactions,total_units_sold,total_revenue
12,31,1880.00


In [0]:
%sql
-- Question B
SELECT 
    store,
    SUM(revenue) AS total_revenue
FROM module1_retail.sales
GROUP BY store
ORDER BY total_revenue DESC, store ASC;

store,total_revenue
Miami,700.00
Hialeah,640.00
Kendall,540.00


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Question C
SELECT 
    category,
    SUM(quantity) AS total_units_sold,
    SUM(revenue) AS total_revenue
FROM module1_retail.sales
GROUP BY category
ORDER BY total_revenue DESC;

category,total_units_sold,total_revenue
Electronics,6,1000.00
Home,11,460.00
Sports,14,420.00


In [0]:
%sql
-- Question D
SELECT 
    transaction_id,
    store,
    category,
    revenue
FROM module1_retail.sales
WHERE revenue >= 200
ORDER BY revenue DESC, transaction_id ASC;

transaction_id,store,category,revenue
6,Hialeah,Electronics,300.00
10,Miami,Electronics,300.00
1,Miami,Electronics,200.00
5,Kendall,Electronics,200.00


### Interpretation & Concepts

1. **Which store generated the most revenue? Cite the value.**  
   Miami generated the most revenue, bringing in a total of $700.00.

2. **Which category sold the most units? Did it also earn the most revenue? Explain.**  
   The Sports category sold the most units with 14 total units sold. Yet, it did not earn the most revenue; Sports generated $420.00, Electronics generated the highest revenue at $1,000.00 selling 6 units. This occurred because Electronics has a significantly higher price per unit ($100.00–$200.00) compared to Sports items ($25.00–$30.00).

3. **How does a volume differ from a table?**  
   In Databricks Unity Catalog, a volume is a storage object used to hold unstructured or semi-structured raw data files (such as CSVs, JSONs, or images), whereas a table stores structured data in a tabular schema (rows and columns) that can be queried directly using SQL.

4. **What does serverless compute do in this activity?**  
   Serverless compute automatically provisions and manages the underlying cloud infrastructure required to execute SQL queries on demand, allowing us to query data instantly without manually starting, configuring, or maintaining clusters.

5. **Why is this useful preparation for big data work even though the file is small?**  
   Working with a small dataset in Databricks uses the exact same SQL syntax, storage structures (volumes and tables), governance controls, and execution platform as multi-terabyte big data environments. Learning these workflows on a smaller scale builds the foundational skills needed to write scalable queries and manage data architecture at enterprise scale.